# 1. For 5 seed yolov7 tiny weights, calculating mean +- std

In [1]:
import subprocess
import re
import pandas as pd
import numpy as np
import sys
import os

# --- Configuration ---
seeds = [0, 1, 2, 3, 4]
data_yaml = "data/DUP_bal_TrainVal.yaml"
img_size = 320
conf_thres = 0.001
iou_thres = 0.65  # Standardized threshold
device = "0"      # GPU device

# Updated path pattern for YOLOv7-tiny
def get_weight_path(seed):
    return f"runs/train/2_yolov7_tiny_seed_{seed}/weights/last.pt"

# --- Storage for results ---
all_results = []

print(f"Starting YOLOv7-tiny evaluation for {len(seeds)} seeds...")
print("-" * 60)

for seed in seeds:
    weight_path = get_weight_path(seed)
    
    # Check if weight file exists to avoid crashes
    if not os.path.exists(weight_path):
        print(f"Error: Weight file not found at {weight_path}")
        continue

    print(f"Processing Seed {seed}: {weight_path}")
    
    # Construct the YOLOv7 test command
    command = [
        "python", "test.py",
        "--weights", weight_path,
        "--data", data_yaml,
        "--img-size", str(img_size),
        "--conf-thres", str(conf_thres),
        "--iou-thres", str(iou_thres),
        "--task", "test",
        "--device", device,
        "--name", f"eval_yolov7_tiny_seed_{seed}"
    ]
    
    try:
        # Run the command and capture output
        result = subprocess.run(
            command, 
            capture_output=True, 
            text=True, 
            encoding='utf-8'
        )
        
        # Parse the output
        output_text = result.stdout + result.stderr
        
        # Regex to find the result lines (Class, Images, Instances, P, R, mAP50, mAP50-95)
        pattern = r'\s+(all|U|D|P)\s+(\d+)\s+(\d+)\s+([0-9.]+)\s+([0-9.]+)\s+([0-9.]+)\s+([0-9.]+)'
        
        matches = re.findall(pattern, output_text)
        
        if not matches:
            print(f"Warning: No metrics found for seed {seed}.")
            if result.returncode != 0:
                print(f"Command Error:\n{result.stderr}")
        
        for match in matches:
            cls_name, imgs, insts, p, r, map50, map95 = match
            all_results.append({
                "Seed": seed,
                "Class": cls_name,
                "Images": int(imgs),
                "Instances": int(insts),
                "Precision": float(p),
                "Recall": float(r),
                "mAP50": float(map50),
                "mAP50-95": float(map95)
            })
            
        print(f"Seed {seed} completed.")

    except Exception as e:
        print(f"An exception occurred processing seed {seed}: {e}")

print("-" * 60)
print("Evaluation Complete. Processing Data...")

# --- Data Processing ---
if not all_results:
    print("No results were captured. Please check paths and commands.")
    sys.exit(1)

df = pd.DataFrame(all_results)

# 1. Save Raw Output to CSV
csv_filename = "yolov7_tiny_evaluation_results.csv"
df.to_csv(csv_filename, index=False)
print(f"\nRaw results saved to {csv_filename}")

# 2. Calculate Mean and Std
stats = df.groupby('Class')[['mAP50', 'mAP50-95']].agg(['mean', 'std'])

# Map class codes to full names
class_names = {'all': 'All', 'U': 'Ascending', 'D': 'Descending', 'P': 'Passing'}

print("\n" + "="*80)
print(f"{'YOLOv7-tiny FINAL RESULTS (Mean ± Std)':^80}")
print("="*80)
print(f"{'Class':<15} | {'mAP@0.5':<25} | {'mAP@0.5:0.95':<25}")
print("-" * 80)

for code, name in class_names.items():
    if code in stats.index:
        # Get stats
        m50_mean = stats.loc[code, ('mAP50', 'mean')]
        m50_std = stats.loc[code, ('mAP50', 'std')]
        
        m95_mean = stats.loc[code, ('mAP50-95', 'mean')]
        m95_std = stats.loc[code, ('mAP50-95', 'std')]
        
        # Format strings (e.g., "97.60 ± 0.30 %")
        res_50 = f"{m50_mean*100:.2f} ± {m50_std*100:.2f} %"
        res_95 = f"{m95_mean*100:.2f} ± {m95_std*100:.2f} %"
        
        print(f"{name:<15} | {res_50:<25} | {res_95:<25}")

print("="*80)

Starting YOLOv7-tiny evaluation for 5 seeds...
------------------------------------------------------------
Processing Seed 0: runs/train/2_yolov7_tiny_seed_0/weights/last.pt
Seed 0 completed.
Processing Seed 1: runs/train/2_yolov7_tiny_seed_1/weights/last.pt
Seed 1 completed.
Processing Seed 2: runs/train/2_yolov7_tiny_seed_2/weights/last.pt
Seed 2 completed.
Processing Seed 3: runs/train/2_yolov7_tiny_seed_3/weights/last.pt
Seed 3 completed.
Processing Seed 4: runs/train/2_yolov7_tiny_seed_4/weights/last.pt
Seed 4 completed.
------------------------------------------------------------
Evaluation Complete. Processing Data...

Raw results saved to yolov7_tiny_evaluation_results.csv

                     YOLOv7-tiny FINAL RESULTS (Mean ± Std)                     
Class           | mAP@0.5                   | mAP@0.5:0.95             
--------------------------------------------------------------------------------
All             | 97.58 ± 0.29 %            | 80.72 ± 1.11 %           
As

# 2. SD net output

In [ ]:
import subprocess
import re
import pandas as pd
import numpy as np
import sys
import os

# --- Configuration ---
seeds = [0, 1, 2, 3, 4]
data_yaml  = "data/DUP_bal_TrainVal.yaml"
img_size   = 320
conf_thres = 0.001
iou_thres  = 0.65  # Standardized threshold
device     = "0"      # GPU device

# UPDATE THIS PATH PATTERN based on where your weights are stored relative to the script
def get_weight_path(seed):
    return f"runs/train/SD_net_TrainVal_2h6a_from_scratch_320_seed{seed}/weights/last.pt"

# --- Storage for results ---
all_results = []

print(f"Starting SD-Net evaluation for {len(seeds)} seeds...")
print("-" * 60)

for seed in seeds:
    weight_path = get_weight_path(seed)
    
    # Check if weight file exists to avoid crashes
    if not os.path.exists(weight_path):
        print(f"Error: Weight file not found at {weight_path}")
        continue

    print(f"Processing Seed {seed}: {weight_path}")
    
    # Construct the test command for SD-Net
    command = [
        "python", "test.py",
        "--weights", weight_path,
        "--data", data_yaml,
        "--img-size", str(img_size),
        "--conf-thres", str(conf_thres),
        "--iou-thres", str(iou_thres),
        "--task", "test",
        "--device", device,
        "--name", f"eval_sdnet_seed_{seed}"
    ]
    
    try:
        # Run the command and capture output
        result = subprocess.run(
            command, 
            capture_output=True, 
            text=True, 
            encoding='utf-8'
        )
        
        # Parse the output
        output_text = result.stdout + result.stderr
        
        # Regex to find the result lines (Class, Images, Instances, P, R, mAP50, mAP50-95)
        pattern = r'\s+(all|U|D|P)\s+(\d+)\s+(\d+)\s+([0-9.]+)\s+([0-9.]+)\s+([0-9.]+)\s+([0-9.]+)'
        
        matches = re.findall(pattern, output_text)
        
        if not matches:
            print(f"Warning: No metrics found for seed {seed}.")
            # Print error if command failed
            if result.returncode != 0:
                print(f"Command Error:\n{result.stderr}")
        
        for match in matches:
            cls_name, imgs, insts, p, r, map50, map95 = match
            all_results.append({
                "Seed": seed,
                "Class": cls_name,
                "Images": int(imgs),
                "Instances": int(insts),
                "Precision": float(p),
                "Recall": float(r),
                "mAP50": float(map50),
                "mAP50-95": float(map95)
            })
            
        print(f"Seed {seed} completed.")

    except Exception as e:
        print(f"An exception occurred processing seed {seed}: {e}")

print("-" * 60)
print("Evaluation Complete. Processing Data...")

# --- Data Processing ---
if not all_results:
    print("No results were captured. Please check paths and commands.")
    sys.exit(1)

df = pd.DataFrame(all_results)

# 1. Save Raw Output to CSV
csv_filename = "sdnet_evaluation_results.csv"
df.to_csv(csv_filename, index=False)
print(f"\nRaw results saved to {csv_filename}")

# 2. Calculate Mean and Std
stats = df.groupby('Class')[['mAP50', 'mAP50-95']].agg(['mean', 'std'])

# Map class codes to full names
class_names = {'all': 'All', 'U': 'Ascending', 'D': 'Descending', 'P': 'Passing'}

print("\n" + "="*80)
print(f"{'SD-Net FINAL RESULTS (Mean ± Std)':^80}")
print("="*80)
print(f"{'Class':<15} | {'mAP@0.5':<25} | {'mAP@0.5:0.95':<25}")
print("-" * 80)

for code, name in class_names.items():
    if code in stats.index:
        # Get stats
        m50_mean = stats.loc[code, ('mAP50', 'mean')]
        m50_std = stats.loc[code, ('mAP50', 'std')]
        
        m95_mean = stats.loc[code, ('mAP50-95', 'mean')]
        m95_std = stats.loc[code, ('mAP50-95', 'std')]
        
        # Format strings (e.g., "97.60 ± 0.30 %")
        res_50 = f"{m50_mean*100:.2f} ± {m50_std*100:.2f} %"
        res_95 = f"{m95_mean*100:.2f} ± {m95_std*100:.2f} %"
        
        print(f"{name:<15} | {res_50:<25} | {res_95:<25}")

print("="*80)

Starting SD-Net evaluation for 5 seeds...
------------------------------------------------------------
Processing Seed 0: runs/train/SD_net_TrainVal_2h6a_from_scratch_320_seed0/weights/last.pt
Seed 0 completed.
Processing Seed 1: runs/train/SD_net_TrainVal_2h6a_from_scratch_320_seed1/weights/last.pt
Seed 1 completed.
Processing Seed 2: runs/train/SD_net_TrainVal_2h6a_from_scratch_320_seed2/weights/last.pt
Seed 2 completed.
Processing Seed 3: runs/train/SD_net_TrainVal_2h6a_from_scratch_320_seed3/weights/last.pt
Seed 3 completed.
Processing Seed 4: runs/train/SD_net_TrainVal_2h6a_from_scratch_320_seed4/weights/last.pt
Seed 4 completed.
------------------------------------------------------------
Evaluation Complete. Processing Data...

Raw results saved to sdnet_evaluation_results.csv

                       SD-Net FINAL RESULTS (Mean ± Std)                        
Class           | mAP@0.5                   | mAP@0.5:0.95             
---------------------------------------------------

In [4]:
!python test.py --weights runs/train/2_yolov7_tiny_seed_0/weights/last.pt --data data/DUP_data.yaml --task val --img-size 320 --conf-thres 0.001 --iou-thres 0.65 --save-json --name v7_tiny_seed_0
!python test.py --weights runs/train/2_yolov7_tiny_seed_1/weights/last.pt --data data/DUP_data.yaml --task val --img-size 320 --conf-thres 0.001 --iou-thres 0.65 --save-json --name v7_tiny_seed_1
!python test.py --weights runs/train/2_yolov7_tiny_seed_2/weights/last.pt --data data/DUP_data.yaml --task val --img-size 320 --conf-thres 0.001 --iou-thres 0.65 --save-json --name v7_tiny_seed_2
!python test.py --weights runs/train/2_yolov7_tiny_seed_3/weights/last.pt --data data/DUP_data.yaml --task val --img-size 320 --conf-thres 0.001 --iou-thres 0.65 --save-json --name v7_tiny_seed_3
!python test.py --weights runs/train/2_yolov7_tiny_seed_4/weights/last.pt --data data/DUP_data.yaml --task val --img-size 320 --conf-thres 0.001 --iou-thres 0.65 --save-json --name v7_tiny_seed_4

Namespace(weights=['runs/train/2_yolov7_tiny_seed_0/weights/last.pt'], data='data/DUP_data.yaml', batch_size=32, img_size=320, conf_thres=0.001, iou_thres=0.65, task='val', device='', single_cls=False, augment=False, verbose=False, save_txt=False, save_hybrid=False, save_conf=False, save_json=True, project='runs/test', name='v7_tiny_seed_0', exist_ok=False, no_trace=False, v5_metric=False)
YOLOR 🚀 998d7201 torch 2.9.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4080, 15926.6875MB)

Fusing layers... 
IDetect.fuse
Model Summary: 208 layers, 6013008 parameters, 0 gradients
 Convert model to Traced-model... 
 traced_script_module saved! 
 model is traced! 

/home/hj/anaconda3/envs/hj/lib/python3.13/site-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4317.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
val: Scanning '

In [ ]:
!python test.py --weights runs/train/SD_net_TrainVal_2h6a_from_scratch_320_seed0/weights/last.pt --data data/DUP_data.yaml --task val --img-size 320 --conf-thres 0.001 --iou-thres 0.65 --save-json --name sd_net_seed_0
!python test.py --weights runs/train/SD_net_TrainVal_2h6a_from_scratch_320_seed1/weights/last.pt --data data/DUP_data.yaml --task val --img-size 320 --conf-thres 0.001 --iou-thres 0.65 --save-json --name sd_net_seed_1
!python test.py --weights runs/train/SD_net_TrainVal_2h6a_from_scratch_320_seed2/weights/last.pt --data data/DUP_data.yaml --task val --img-size 320 --conf-thres 0.001 --iou-thres 0.65 --save-json --name sd_net_seed_2
!python test.py --weights runs/train/SD_net_TrainVal_2h6a_from_scratch_320_seed3/weights/last.pt --data data/DUP_data.yaml --task val --img-size 320 --conf-thres 0.001 --iou-thres 0.65 --save-json --name sd_net_seed_3
!python test.py --weights runs/train/SD_net_TrainVal_2h6a_from_scratch_320_seed4/weights/last.pt --data data/DUP_data.yaml --task val --img-size 320 --conf-thres 0.001 --iou-thres 0.65 --save-json --name sd_net_seed_4

Namespace(weights=['runs/train/SD_net_TrainVal_2h6a_from_scratch_320_seed0/weights/last.pt'], data='data/DUP_data.yaml', batch_size=32, img_size=320, conf_thres=0.001, iou_thres=0.65, task='val', device='', single_cls=False, augment=False, verbose=False, save_txt=False, save_hybrid=False, save_conf=False, save_json=True, project='runs/test', name='sd_net_seed_0', exist_ok=False, no_trace=False, v5_metric=False)
YOLOR 🚀 998d7201 torch 2.9.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4080, 15926.6875MB)

Fusing layers... 
IDetect.fuse
Model Summary: 132 layers, 1588064 parameters, 0 gradients
 Convert model to Traced-model... 
 traced_script_module saved! 
 model is traced! 

/home/hj/anaconda3/envs/hj/lib/python3.13/site-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4317.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-de

In [11]:
!python test.py --weights runs/train/2_yolov7_tiny_seed_0/weights/last.pt --data data/DUP_data.yaml --task val --img-size 320 --conf-thres 0.001 --iou-thres 0.65 --name v7_tiny_csv_seed_0
!python test.py --weights runs/train/2_yolov7_tiny_seed_1/weights/last.pt --data data/DUP_data.yaml --task val --img-size 320 --conf-thres 0.001 --iou-thres 0.65 --name v7_tiny_csv_seed_1
!python test.py --weights runs/train/2_yolov7_tiny_seed_2/weights/last.pt --data data/DUP_data.yaml --task val --img-size 320 --conf-thres 0.001 --iou-thres 0.65 --name v7_tiny_csv_seed_2
!python test.py --weights runs/train/2_yolov7_tiny_seed_3/weights/last.pt --data data/DUP_data.yaml --task val --img-size 320 --conf-thres 0.001 --iou-thres 0.65 --name v7_tiny_csv_seed_3
!python test.py --weights runs/train/2_yolov7_tiny_seed_4/weights/last.pt --data data/DUP_data.yaml --task val --img-size 320 --conf-thres 0.001 --iou-thres 0.65 --name v7_tiny_csv_seed_4

!python test.py --weights runs/train/SD_net_TrainVal_2h6a_from_scratch_320_seed0/weights/last.pt --data data/DUP_data.yaml --task val --img-size 320 --conf-thres 0.001 --iou-thres 0.65 --name sd_net_csv_seed_0
!python test.py --weights runs/train/SD_net_TrainVal_2h6a_from_scratch_320_seed1/weights/last.pt --data data/DUP_data.yaml --task val --img-size 320 --conf-thres 0.001 --iou-thres 0.65 --name sd_net_csv_seed_1
!python test.py --weights runs/train/SD_net_TrainVal_2h6a_from_scratch_320_seed2/weights/last.pt --data data/DUP_data.yaml --task val --img-size 320 --conf-thres 0.001 --iou-thres 0.65 --name sd_net_csv_seed_2
!python test.py --weights runs/train/SD_net_TrainVal_2h6a_from_scratch_320_seed3/weights/last.pt --data data/DUP_data.yaml --task val --img-size 320 --conf-thres 0.001 --iou-thres 0.65 --name sd_net_csv_seed_3
!python test.py --weights runs/train/SD_net_TrainVal_2h6a_from_scratch_320_seed4/weights/last.pt --data data/DUP_data.yaml --task val --img-size 320 --conf-thres 0.001 --iou-thres 0.65 --name sd_net_csv_seed_4

Namespace(weights=['runs/train/2_yolov7_tiny_seed_0/weights/last.pt'], data='data/DUP_data.yaml', batch_size=32, img_size=320, conf_thres=0.001, iou_thres=0.65, task='val', device='', single_cls=False, augment=False, verbose=False, save_txt=False, save_hybrid=False, save_conf=False, save_json=False, project='runs/test', name='v7_tiny_csv_seed_0', exist_ok=False, no_trace=False, v5_metric=False)
YOLOR 🚀 998d7201 torch 2.9.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4080, 15926.6875MB)

Fusing layers... 
IDetect.fuse
Model Summary: 208 layers, 6013008 parameters, 0 gradients
 Convert model to Traced-model... 
 traced_script_module saved! 
 model is traced! 

/home/hj/anaconda3/envs/hj/lib/python3.13/site-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4317.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
val: Scann

In [13]:
import json
import numpy as np
import contextlib
import io
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

# 1. Ground Truth File Path
annFile = '/mnt/Documents/Dad/github/DUP/yolo_to_coco/output/test.json'

# 2. Define model paths for all seeds
models_to_evaluate = {
    "v7_tiny": [f'runs/test/v7_tiny_seed_{i}/last_predictions.json' for i in range(5)],
    "sd_net": [f'runs/test/sd_net_seed_{i}/last_predictions.json' for i in range(5)]
}

# 3. Load ground truth and create mapping
print("Loading annotations...")
cocoGt = COCO(annFile)
name_to_id = {img['file_name'].split('/')[-1].split('.')[0]: img['id'] for img in cocoGt.dataset['images']}

categories = ['mAP', 'mAP_50', 'mAP_75', 'mAP_s', 'mAP_m', 'mAP_l']

# 4. Loop through models and seeds
for model_name, res_files in models_to_evaluate.items():
    print(f"\nProcessing {model_name}...")
    model_stats = []
    
    for i, resFile in enumerate(res_files):
        try:
            with open(resFile, 'r') as f:
                preds = json.load(f)
        except FileNotFoundError:
            print(f"  [Warning] Seed {i} file not found: {resFile}")
            continue

        # Fix image_ids (YOLO string -> COCO integer)
        fixed_preds = []
        for p in preds:
            orig_id = str(p['image_id'])
            if orig_id in name_to_id:
                p['image_id'] = name_to_id[orig_id]
                # p['category_id'] += 1 # Uncomment if results are all 0.00
                fixed_preds.append(p)

        # Evaluate through COCO API
        with contextlib.redirect_stdout(io.StringIO()): # Suppress intermediate prints
            cocoDt = cocoGt.loadRes(fixed_preds)
            cocoEval = COCOeval(cocoGt, cocoDt, 'bbox')
            cocoEval.evaluate()
            cocoEval.accumulate()
            cocoEval.summarize()
        
        # Store metrics (mAP, mAP_50, mAP_75, mAP_s, mAP_m, mAP_l)
        # Multiply by 100 for percentage
        model_stats.append(cocoEval.stats[0:6] * 100)
        print(f"  Seed {i} evaluated.")

    # 5. Calculate and print mean +- std
    if model_stats:
        results_matrix = np.array(model_stats)
        means = np.mean(results_matrix, axis=0)
        # CHANGED: ddof=0 to match the Population Std used in your LaTeX table
        stds = np.std(results_matrix, axis=0, ddof=0) 

        print(f"\n--- {model_name.upper()} Final Results (%) ---")
        print(f"{'Metric':<8} | {'Mean ± Std':<15}")
        print("-" * 26)
        for cat, m, s in zip(categories, means, stds):
            print(f"{cat:<8} | {m:.2f} ± {s:.2f}")
    else:
        print(f"No results found for {model_name}.")

Loading annotations...
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!

Processing v7_tiny...
  Seed 0 evaluated.
  Seed 1 evaluated.
  Seed 2 evaluated.
  Seed 3 evaluated.
  Seed 4 evaluated.

--- V7_TINY Final Results (%) ---
Metric   | Mean ± Std     
--------------------------
mAP      | 80.32 ± 0.96
mAP_50   | 97.08 ± 0.22
mAP_75   | 92.47 ± 0.52
mAP_s    | 35.34 ± 2.85
mAP_m    | 72.38 ± 1.60
mAP_l    | 85.41 ± 0.72

Processing sd_net...
  Seed 0 evaluated.
  Seed 1 evaluated.
  Seed 2 evaluated.
  Seed 3 evaluated.
  Seed 4 evaluated.

--- SD_NET Final Results (%) ---
Metric   | Mean ± Std     
--------------------------
mAP      | 75.78 ± 0.41
mAP_50   | 95.73 ± 0.25
mAP_75   | 89.90 ± 0.68
mAP_s    | 21.23 ± 3.54
mAP_m    | 67.81 ± 0.87
mAP_l    | 80.99 ± 0.33


In [7]:
import json
import numpy as np
import contextlib
import io
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

# 1. Ground Truth File Path
annFile = '/mnt/Documents/Dad/github/DUP/yolo_to_coco/output/test.json'

# 2. Define model paths for all seeds
models_to_evaluate = {
    "v7_tiny": [f'runs/test/v7_tiny_seed_{i}/last_predictions.json' for i in range(5)],
    "sd_net": [f'runs/test/sd_net_seed_{i}/last_predictions.json' for i in range(5)]
}

# 3. Load ground truth and create mapping
print("Loading annotations...")
cocoGt = COCO(annFile)
name_to_id = {img['file_name'].split('/')[-1].split('.')[0]: img['id'] for img in cocoGt.dataset['images']}

# Only these categories will be printed
display_categories = ['mAP_small', 'mAP_medium', 'mAP_large']

# 4. Loop through models and seeds
for model_name, res_files in models_to_evaluate.items():
    print(f"\nProcessing {model_name}...")
    model_stats = []
    
    for i, resFile in enumerate(res_files):
        try:
            with open(resFile, 'r') as f:
                preds = json.load(f)
        except FileNotFoundError:
            print(f"  [Warning] Seed {i} file not found: {resFile}")
            continue

        # Fix image_ids (YOLO string -> COCO integer)
        fixed_preds = []
        for p in preds:
            orig_id = str(p['image_id'])
            if orig_id in name_to_id:
                p['image_id'] = name_to_id[orig_id]
                # p['category_id'] += 1 # Uncomment if results are all 0.00
                fixed_preds.append(p)

        # Evaluate through COCO API
        with contextlib.redirect_stdout(io.StringIO()): # Suppress intermediate prints
            cocoDt = cocoGt.loadRes(fixed_preds)
            cocoEval = COCOeval(cocoGt, cocoDt, 'bbox')
            cocoEval.evaluate()
            cocoEval.accumulate()
            cocoEval.summarize()
        
        # Store all 6 standard COCO metrics (indices 0-5)
        model_stats.append(cocoEval.stats[0:6] * 100)
        print(f"  Seed {i} evaluated.")

    # 5. Calculate and print mean +- std
    if model_stats:
        results_matrix = np.array(model_stats)
        means = np.mean(results_matrix, axis=0)
        # ddof=0 for Population Std to match your table exactly
        stds = np.std(results_matrix, axis=0, ddof=0) 

        print(f"\n--- {model_name.upper()} Scale Results (%) ---")
        print(f"{'Metric':<12} | {'Mean ± Std':<15}")
        print("-" * 30)
        
        # Slicing from index 3 to 5 to get Small, Medium, and Large metrics
        for idx, cat in enumerate(display_categories):
            # Scale metrics are at indices 3, 4, and 5 in the COCO stats array
            m = means[idx + 3]
            s = stds[idx + 3]
            print(f"{cat:<12} | {m:.2f} ± {s:.2f}")
    else:
        print(f"No results found for {model_name}.")

Loading annotations...
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!

Processing v7_tiny...
  Seed 0 evaluated.
  Seed 1 evaluated.
  Seed 2 evaluated.
  Seed 3 evaluated.
  Seed 4 evaluated.

--- V7_TINY Scale Results (%) ---
Metric       | Mean ± Std     
------------------------------
mAP_small    | 35.34 ± 2.85
mAP_medium   | 72.38 ± 1.60
mAP_large    | 85.41 ± 0.72

Processing sd_net...
  Seed 0 evaluated.
  Seed 1 evaluated.
  Seed 2 evaluated.
  Seed 3 evaluated.
  Seed 4 evaluated.

--- SD_NET Scale Results (%) ---
Metric       | Mean ± Std     
------------------------------
mAP_small    | 21.23 ± 3.54
mAP_medium   | 67.81 ± 0.87
mAP_large    | 80.99 ± 0.33
